# 03 - Inference
Load the saved pipeline (`models/model.joblib`) and predict. The same file is used inside the Docker image.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings("ignore")

In [2]:
import joblib, json, pandas as pd
from heart import config
model = joblib.load(ROOT / 'models' / 'model.joblib')
sample = json.load(open(ROOT / 'sample_request.json'))
proba = model.predict_proba(pd.DataFrame([sample])[config.FEATURES])[0, 1]
print(f'probability of heart disease = {proba:.3f} -> prediction = {int(proba >= 0.5)}')

probability of heart disease = 0.993 -> prediction = 1


## Batch prediction on the cleaned dataset

In [3]:
from heart.data import load_clean
df = load_clean()
df['proba'] = model.predict_proba(df[config.FEATURES])[:, 1].round(3)
df['pred'] = (df['proba'] >= 0.5).astype(int)
print('accuracy on full data:', (df.pred == df.target).mean().round(3))
df[['age','sex','cp','thalach','oldpeak','target','proba','pred']].head(10)

accuracy on full data: 0.861


,age,sex,cp,thalach,oldpeak,target,proba,pred
0,63.0,1,1,150.0,2.3,0,0.267,0
1,67.0,1,4,108.0,1.5,1,0.995,1
2,67.0,1,4,129.0,2.6,1,0.990,1
3,37.0,1,3,187.0,3.5,0,0.115,0
4,41.0,0,2,172.0,1.4,0,0.057,0
5,56.0,1,2,178.0,0.8,0,0.063,0
6,62.0,0,4,160.0,3.6,1,0.798,1
7,57.0,0,4,163.0,0.6,0,0.132,0
8,63.0,1,4,147.0,1.4,1,0.934,1
9,53.0,1,4,155.0,3.1,1,0.835,1


## Calling the REST API
When the container is running (`docker run -p 8080:8000 heart-api:1.0.0`):

In [4]:
import urllib.request
try:
    req = urllib.request.Request('http://127.0.0.1:8080/predict', data=json.dumps(sample).encode(), headers={'Content-Type': 'application/json'})
    print(urllib.request.urlopen(req, timeout=3).read().decode())
except Exception as e:
    print('API not running:', e)

{"prediction":1,"label":"heart disease","confidence":0.9934,"probability_disease":0.9934,"risk_level":"high","model_version":"logistic_regression-c800af5b"}
